In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, window, count, sum as _sum, avg,
    min as _min, max as _max, round as _round, desc,  to_timestamp
)

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [5]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+-------+-----------+--------+--------------------------+------+-------+
|amount |category   |store   |timestamp                 |tx_id |user_id|
+-------+-----------+--------+--------------------------+------+-------+
|3268.45|odzież     |Wrocław |2026-05-04T20:30:10.998447|TX1595|u02    |
|850.54 |odzież     |Wrocław |2026-05-04T20:30:12.002605|TX5678|u19    |
|2472.61|książki    |Kraków  |2026-05-04T20:30:13.003128|TX3016|u19    |
|1378.57|elektronika|Wrocław |2026-05-04T20:30:14.005233|TX6397|u19    |
|2099.71|książki    |Warszawa|2026-05-04T20:30:15.007831|TX5279|u08    |
|3020.04|odzież     |Gdańsk  |2026-05-04T20:30:16.008261|TX3771|u08    |
|2549.24|odzież     |Wrocław |2026-05-04T20:30:17.009443|TX7998|u06    |
|3864.33|

In [6]:
df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

In [7]:
df.groupBy("store") \
  .agg(
      count("tx_id").alias("liczba_tx"),
      _round(_sum("amount"), 2).alias("suma_PLN"),
      _round(avg("amount"), 2).alias("srednia_PLN"),
  ) \
  .orderBy("store") \
  .show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2436|6101768.01|    2504.83|
|  Kraków|     2529|6387482.59|    2525.69|
|Warszawa|     2537|6390417.09|    2518.89|
| Wrocław|     2498| 6205809.2|    2484.31|
+--------+---------+----------+-----------+



In [8]:
df.groupBy("category") \
  .agg(
      _round(_sum("amount"), 2).alias("suma_PLN"),
      _min("amount").alias("min_PLN"),
      _max("amount").alias("max_PLN"),
  ) \
  .orderBy("category") \
  .show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|6182311.48|   5.86| 4999.5|
|    książki|6410139.21|   5.78|4999.06|
|     odzież|6327871.38|   5.43|4999.84|
|    żywność|6165154.82|   6.08|4996.98|
+-----------+----------+-------+-------+



In [9]:
df.groupBy(window("timestamp", "1 hour")) \
  .agg(
      count("tx_id").alias("liczba_tx"),
      _round(_sum("amount"), 2).alias("suma_PLN"),
  ) \
  .orderBy("window") \
  .show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-04 20:00:00, 2026-05-04 21:00:00}|1788     |4601332.37|
|{2026-05-04 21:00:00, 2026-05-04 22:00:00}|3598     |8932219.77|
|{2026-05-04 22:00:00, 2026-05-04 23:00:00}|564      |1409896.16|
|{2026-05-05 17:00:00, 2026-05-05 18:00:00}|1859     |4631256.31|
|{2026-05-05 18:00:00, 2026-05-05 19:00:00}|2191     |5510772.28|
+------------------------------------------+---------+----------+



In [10]:
df.groupBy(window("timestamp", "30 minutes"), "store") \
  .agg(
      count("tx_id").alias("liczba_tx"),
      _round(_sum("amount"), 2).alias("suma_PLN"),
  ) \
  .orderBy("window", "store") \
  .show(truncate=False)

+------------------------------------------+--------+---------+----------+
|window                                    |store   |liczba_tx|suma_PLN  |
+------------------------------------------+--------+---------+----------+
|{2026-05-04 20:30:00, 2026-05-04 21:00:00}|Gdańsk  |422      |1102262.14|
|{2026-05-04 20:30:00, 2026-05-04 21:00:00}|Kraków  |428      |1079806.44|
|{2026-05-04 20:30:00, 2026-05-04 21:00:00}|Warszawa|455      |1202710.36|
|{2026-05-04 20:30:00, 2026-05-04 21:00:00}|Wrocław |483      |1216553.43|
|{2026-05-04 21:00:00, 2026-05-04 21:30:00}|Gdańsk  |403      |976841.31 |
|{2026-05-04 21:00:00, 2026-05-04 21:30:00}|Kraków  |505      |1279180.9 |
|{2026-05-04 21:00:00, 2026-05-04 21:30:00}|Warszawa|457      |1123187.13|
|{2026-05-04 21:00:00, 2026-05-04 21:30:00}|Wrocław |434      |1099394.18|
|{2026-05-04 21:30:00, 2026-05-04 22:00:00}|Gdańsk  |478      |1164171.75|
|{2026-05-04 21:30:00, 2026-05-04 22:00:00}|Kraków  |446      |1137444.22|
|{2026-05-04 21:30:00, 20

In [11]:
df.filter(col("store") == "Kraków") \
  .groupBy(window("timestamp", "1 hour")) \
  .agg(_round(_sum("amount"), 2).alias("suma_PLN")) \
  .orderBy(desc("suma_PLN")) \
  .show(truncate=False)

+------------------------------------------+----------+
|window                                    |suma_PLN  |
+------------------------------------------+----------+
|{2026-05-04 21:00:00, 2026-05-04 22:00:00}|2416625.12|
|{2026-05-05 18:00:00, 2026-05-05 19:00:00}|1403564.75|
|{2026-05-05 17:00:00, 2026-05-05 18:00:00}|1157742.9 |
|{2026-05-04 20:00:00, 2026-05-04 21:00:00}|1079806.44|
|{2026-05-04 22:00:00, 2026-05-04 23:00:00}|329743.38 |
+------------------------------------------+----------+



In [12]:
df.groupBy(window("timestamp", "1 hour", "30 minutes")) \
  .agg(
      count("tx_id").alias("liczba_tx"),
      _round(_sum("amount"), 2).alias("suma_PLN"),
  ) \
  .orderBy("window") \
  .show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-04 20:00:00, 2026-05-04 21:00:00}|1788     |4601332.37|
|{2026-05-04 20:30:00, 2026-05-04 21:30:00}|3587     |9079935.89|
|{2026-05-04 21:00:00, 2026-05-04 22:00:00}|3598     |8932219.77|
|{2026-05-04 21:30:00, 2026-05-04 22:30:00}|2363     |5863512.41|
|{2026-05-04 22:00:00, 2026-05-04 23:00:00}|564      |1409896.16|
|{2026-05-05 16:30:00, 2026-05-05 17:30:00}|61       |122919.2  |
|{2026-05-05 17:00:00, 2026-05-05 18:00:00}|1859     |4631256.31|
|{2026-05-05 17:30:00, 2026-05-05 18:30:00}|3596     |9013710.87|
|{2026-05-05 18:00:00, 2026-05-05 19:00:00}|2191     |5510772.28|
|{2026-05-05 18:30:00, 2026-05-05 19:30:00}|393      |1005398.52|
+------------------------------------------+---------+----------+



In [13]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)

sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)

print(f"Tumbling (1h): {tumbling_rows}")
print(f"Sliding (1h/30min): {sliding_rows}")

# ODPOWIEDŹ:
# Sliding ma więcej wierszy, ponieważ okna się nakładają, a jedna transakcja może należeć do wielu okien.

Tumbling (1h): 5
Sliding (1h/30min): 10


In [14]:
#Pytanie kontrolne
# 1. Ile transakcji jest w oknie 09:00–10:00?
# # W danych nie występuje okno 09:00–10:00.
# Najbliższe dostępne okna to:
# 20:00–21:00, 21:00–22:00 itd.

# 2. aka jest różnica między groupBy("store") a groupBy(window(...), "store")?
# groupBy(store): globalna agregacja bez czasu
# groupBy(window, store): agregacja czasowa + lokalizacja

# 3.  W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?:
# transakcja powinna należeć do 2 okien (09:00 i 09:30)

In [15]:
df.filter(col("store") == "Gdańsk") \
  .groupBy(window("timestamp", "1 hour")) \
  .agg(_round(avg("amount"), 2).alias("avg")) \
  .orderBy("avg") \
  .show()

+--------------------+-------+
|              window|    avg|
+--------------------+-------+
|{2026-05-04 21:00...|2430.21|
|{2026-05-05 17:00...|2484.97|
|{2026-05-05 18:00...|2524.15|
|{2026-05-04 20:00...| 2612.0|
|{2026-05-04 22:00...|2635.39|
+--------------------+-------+



In [16]:
df.groupBy(window("timestamp", "30 minutes"), "category") \
  .agg(count("tx_id").alias("tx")) \
  .orderBy("window", "category") \
  .show()

+--------------------+-----------+---+
|              window|   category| tx|
+--------------------+-----------+---+
|{2026-05-04 20:30...|elektronika|435|
|{2026-05-04 20:30...|    książki|445|
|{2026-05-04 20:30...|     odzież|470|
|{2026-05-04 20:30...|    żywność|438|
|{2026-05-04 21:00...|elektronika|467|
|{2026-05-04 21:00...|    książki|433|
|{2026-05-04 21:00...|     odzież|451|
|{2026-05-04 21:00...|    żywność|448|
|{2026-05-04 21:30...|elektronika|449|
|{2026-05-04 21:30...|    książki|460|
|{2026-05-04 21:30...|     odzież|450|
|{2026-05-04 21:30...|    żywność|440|
|{2026-05-04 22:00...|elektronika|146|
|{2026-05-04 22:00...|    książki|143|
|{2026-05-04 22:00...|     odzież|157|
|{2026-05-04 22:00...|    żywność|118|
|{2026-05-05 17:00...|elektronika| 16|
|{2026-05-05 17:00...|    książki| 16|
|{2026-05-05 17:00...|     odzież| 14|
|{2026-05-05 17:00...|    żywność| 15|
+--------------------+-----------+---+
only showing top 20 rows



In [17]:
df.groupBy(window("timestamp", "15 minutes")) \
  .agg(count("tx_id").alias("tx")) \
  .orderBy(desc("tx")) \
  .show()

+--------------------+---+
|              window| tx|
+--------------------+---+
|{2026-05-04 21:00...|900|
|{2026-05-04 21:30...|900|
|{2026-05-05 17:30...|899|
|{2026-05-04 21:45...|899|
|{2026-05-04 21:15...|899|
|{2026-05-05 18:15...|899|
|{2026-05-05 18:00...|899|
|{2026-05-05 17:45...|899|
|{2026-05-04 20:45...|899|
|{2026-05-04 20:30...|889|
|{2026-05-04 22:00...|564|
|{2026-05-05 18:30...|393|
|{2026-05-05 17:15...| 61|
+--------------------+---+

